# Task 1, Steps 2-3: P1 Magnitude Pruning and P2 Wanda

Unstructured, per-output-row pruning of every eligible projection at s in {0.25, 0.50, 0.75}.
* **P1:** score |W_ij|.  **P2:** score |W_ij| * ||X_j||_2 with X from the 128 x 512 C4 calibration windows.
* Both walk the blocks in order; block l+1 sees the outputs of the already-pruned block l (as in the Wanda code).
  The magnitude mask does not depend on activations, but this pass also measures its calibration output error.
* Each condition starts from the restored FP16 checkpoint. Only eligible weights are pruned.
* Recorded: achieved nonzero fraction, per-projection calibration output error ||dW X||^2/||W X||^2 (exact, via
  the input Gram matrix), per-block output error, quality, storage, compression time and latency.

HellaSwag: full 2,000 IDs at 50% sparsity, first 512 IDs otherwise.

In [ ]:
# --- Setup: locate (or clone) the PA4 repo, pin library versions, prepare data -------------
import os, sys, subprocess, importlib.metadata as md
REPO_URL = "https://github.com/moizejaz-31/ATML-PA4.git"   # change if your repo lives elsewhere

def _find_root(d):
    d = os.path.abspath(d)
    while True:
        if os.path.isdir(os.path.join(d, "shared")) and os.path.isdir(os.path.join(d, "task1_compression")):
            return d
        if os.path.dirname(d) == d:
            return None
        d = os.path.dirname(d)

ROOT = _find_root(os.getcwd())
if ROOT is None:                                   # fresh Kaggle / Colab runtime
    subprocess.run(["git", "clone", "-q", REPO_URL, "ATML-PA4"], check=True)
    ROOT = os.path.abspath("ATML-PA4")
os.chdir(ROOT)
sys.path.insert(0, ROOT)

def _ver(p):
    try:
        return md.version(p)
    except md.PackageNotFoundError:
        return None

if _ver("transformers") != "4.57.1" or _ver("huggingface_hub") != "0.36.2":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.1", "huggingface_hub==0.36.2",
                    "datasets>=3,<5"], check=True)
    print(">>> Installed pinned versions. Restart the kernel, then run this cell again.")

from shared.src.nb import init_notebook
ctx = init_notebook("task1", need_public=True, need_prefix_banks=False)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from shared.src.plotting import save_figure
from shared.src.io import save_json
from task1_compression.src.runner import setup_task1
from task1_compression.src import prune
from task1_compression.src.model_utils import ELIGIBLE

T = setup_task1(ctx)
SPARSITIES = (0.25, 0.50, 0.75)

In [ ]:
for method, tag in (("magnitude", "p1_mag"), ("wanda", "p2_wanda")):
    for s in SPARSITIES:
        name = f"{tag}_s{int(s*100)}"
        T.runner.run(name, apply_fn=lambda m, method=method, s=s: prune.prune_sequential(
                         m, T.calib, method, s, gram=True, collect_scores=(method == "wanda")),
                     meta=dict(method=method, target_sparsity=s, mask="per-output-row unstructured",
                               calibration="128 x 512 C4 train (seed 6304)"))

## Dense input norms for P3 / OWL (preserved score ingredients)
One dense pass over the calibration windows; saved so P3 can rebuild Wanda scores of the unchanged model.

In [ ]:
norm_path = ctx.checkpoints / "dense_input_norms.pt"
if not norm_path.exists():
    T.snap.restore(T.model)
    norms = prune.dense_input_norms(T.model, T.calib)
    torch.save(norms, norm_path)
print("saved", norm_path)

## Matched-sparsity comparison P1 vs P2

In [ ]:
from task1_compression.src.analysis import summary_table, compare_table
tab = summary_table(ctx.results).loc[["fp16"] + [f"{t}_s{int(s*100)}" for t in ("p1_mag", "p2_wanda") for s in SPARSITIES]]
tab[["sparsity", "wikitext2_ppl", "c4_ppl", "hs_n", "hs_acc", "hs_acc_norm", "sparse16_MB", "compression_s"]]

In [ ]:
compare_table(ctx.results, [(f"p2_wanda_s{int(s*100)}", f"p1_mag_s{int(s*100)}") for s in SPARSITIES]
              + [(f"p2_wanda_s{int(s*100)}", "fp16") for s in SPARSITIES])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for tag, lab in (("p1_mag", "P1 magnitude"), ("p2_wanda", "P2 Wanda")):
    xs = [tab.loc[f"{tag}_s{int(s*100)}", "sparsity"] for s in SPARSITIES]
    axes[0].plot(xs, [tab.loc[f"{tag}_s{int(s*100)}", "wikitext2_ppl"] for s in SPARSITIES], "o-", label=lab)
    axes[1].plot(xs, [tab.loc[f"{tag}_s{int(s*100)}", "c4_ppl"] for s in SPARSITIES], "o-", label=lab)
    axes[2].plot(xs, [tab.loc[f"{tag}_s{int(s*100)}", "hs_acc_norm"] for s in SPARSITIES], "o-", label=lab)
for ax, t in zip(axes, ("WikiText-2 PPL (N=50,000)", "C4 val PPL (N=50,000)", "HellaSwag acc_norm")):
    ax.axhline(tab.loc["fp16", {"WikiText-2 PPL (N=50,000)": "wikitext2_ppl", "C4 val PPL (N=50,000)": "c4_ppl", "HellaSwag acc_norm": "hs_acc_norm"}[t]], color="k", ls="--", lw=1, label="FP16")
    ax.set_xlabel("eligible-weight sparsity"); ax.set_title(t)
axes[0].set_yscale("log"); axes[1].set_yscale("log"); axes[0].legend()
save_figure(ctx.figures, "p1_p2_quality_vs_sparsity")

## Where does removal hurt? Calibration output error per projection

In [ ]:
from task1_compression.src.analysis import load_condition
def err_matrix(name, key="out_err_rel"):
    pp = load_condition(ctx.results, name)["compression"]["stats"]["per_projection"]
    M = np.zeros((28, 7))
    for k, v in pp.items():
        _, li, p = k.split(".")
        M[int(li), ELIGIBLE.index(p)] = v[key]
    return M

fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey=True)
for r, tag in enumerate(("p1_mag", "p2_wanda")):
    for c, s in enumerate(SPARSITIES):
        M = err_matrix(f"{tag}_s{int(s*100)}")
        sns.heatmap(np.log10(M + 1e-12), ax=axes[r, c], cmap="viridis", xticklabels=[p.replace("_proj", "") for p in ELIGIBLE],
                    cbar_kws=dict(label="log10 rel. output error"))
        axes[r, c].set_title(f"{tag} s={s}"); axes[r, c].set_ylabel("block" if c == 0 else "")
save_figure(ctx.figures, "p1_p2_projection_output_error")

In [ ]:
blk = {n: [b["block_out_rel_err"] for b in load_condition(ctx.results, n)["compression"]["stats"]["per_block"]]
       for n in [f"{t}_s{int(s*100)}" for t in ("p1_mag", "p2_wanda") for s in SPARSITIES]}
pd.DataFrame(blk).plot(logy=True, figsize=(9, 4), xlabel="block", ylabel="block output rel. error (calibration)")
save_figure(ctx.figures, "p1_p2_block_output_error")

## Magnitude vs activation (research question 1)
At 50% sparsity: Spearman correlation of |W| and Wanda scores, Jaccard overlap of the two pruned sets, and the
activation-norm outlier ratio max||X_j|| / median||X_j|| for each projection.

In [ ]:
ss = load_condition(ctx.results, "p2_wanda_s50")["compression"]["stats"]["score_stats"]
rows = []
for k, v in ss.items():
    _, li, p = k.split(".")
    rows.append(dict(block=int(li), proj=p, spearman=v["spearman_mag_vs_wanda"], jaccard=v["mask_jaccard_pruned"],
                     act_outlier=v["act_norm_max_over_median"]))
sdf = pd.DataFrame(rows)
sdf.to_csv(ctx.results / "p2_score_relationships_s50.csv", index=False)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ("spearman", "jaccard", "act_outlier")):
    sns.heatmap(sdf.pivot(index="block", columns="proj", values=col)[list(ELIGIBLE)], ax=ax, cmap="magma")
    ax.set_title(col)
save_figure(ctx.figures, "p2_magnitude_vs_activation_s50")
sdf.groupby("proj")[["spearman", "jaccard", "act_outlier"]].mean().loc[list(ELIGIBLE)]

In [ ]:
# How the overlap between magnitude- and Wanda-pruned sets changes with sparsity
jac = {}
for s in SPARSITIES:
    st = load_condition(ctx.results, f"p2_wanda_s{int(s*100)}")["compression"]["stats"]["score_stats"]
    jac[s] = pd.Series({k: v["mask_jaccard_pruned"] for k, v in st.items()}).groupby(lambda k: k.split(".")[2]).mean()
jac = pd.DataFrame(jac).loc[list(ELIGIBLE)]
jac.to_csv(ctx.results / "p2_mask_jaccard_by_sparsity.csv")
jac